# Functional lung segmentation with El Bicho

Segment hyperpolarized-gas ventilation MRI into the model's five classes
(labels 0–4). **Inputs:** a local scalar 3-D ventilation image and an aligned
binary lung mask, supplied through the path cell or environment variables
`ANTSTORCH_EXAMPLE_VENTILATION` and `ANTSTORCH_EXAMPLE_VENTILATION_MASK`.
These are not CT inputs or ordinary proton MRI. **Resources:** model weights
download on first use; prediction batches all slices, so large images can
require substantial memory even when predicting only one slice direction.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = "cpu"
antstorch.set_default_device(device)

In [ ]:
import os
from pathlib import Path

ventilation_path = os.environ.get("ANTSTORCH_EXAMPLE_VENTILATION")
mask_path = os.environ.get("ANTSTORCH_EXAMPLE_VENTILATION_MASK")
# Alternatively set existing absolute paths here.
for name, path in [("ventilation image", ventilation_path), ("lung mask", mask_path)]:
    if path is None or not Path(path).is_file():
        raise FileNotFoundError(f"Supply a local {name} path in this cell or the documented environment variable.")
ventilation = ants.image_read(str(ventilation_path)).clone("float")
lung_mask = ants.image_read(str(mask_path)).clone("float")
for image in (ventilation, lung_mask):
    assert image.dimension == 3 and not image.has_components
    assert np.isfinite(image.numpy()).all()
assert np.ptp(ventilation.numpy()) > 0
assert ventilation.shape == lung_mask.shape
assert np.isin(lung_mask.numpy(), [0, 1]).all() and (lung_mask.numpy() > 0).any()
for attribute in ("spacing", "origin", "direction"):
    np.testing.assert_allclose(getattr(lung_mask, attribute), getattr(ventilation, attribute))
# Matching headers do not establish alignment; inspect the mask before inference.
ants.plot(ventilation, overlay=lung_mask, axis=2, nslices=6,
          overlay_cmap="autumn", overlay_alpha=0.4)

## Predict and inspect the class labels
`use_coarse_slices_only=True` predicts along the axis of greatest slice spacing.
Setting it to False combines all directions at higher computation and memory
cost. Use the label overlay for inspection; retain the probabilities separately.

In [ ]:
result = antstorch.el_bicho(
    ventilation, mask=lung_mask, use_coarse_slices_only=True,
    device=device, verbose=True,
)
segmentation = result["segmentation_image"]
probabilities = result["probability_images"]
assert segmentation.shape == ventilation.shape
assert np.isin(segmentation.numpy(), np.arange(5)).all()
assert len(probabilities) == 5
for attribute in ("spacing", "origin", "direction"):
    np.testing.assert_allclose(getattr(segmentation, attribute), getattr(ventilation, attribute))
for probability in probabilities:
    assert probability.shape == ventilation.shape and np.isfinite(probability.numpy()).all()
# Restrict the displayed/reported regions to the supplied lung mask explicitly.
labels_in_lung = segmentation * lung_mask
ants.plot(ventilation, overlay=labels_in_lung, axis=2, nslices=8,
          overlay_cmap="jet", overlay_alpha=0.6)
voxel_volume = float(np.prod(ventilation.spacing))
lung_voxels = int(np.count_nonzero(lung_mask.numpy()))
for label in range(5):
    count = np.count_nonzero((segmentation.numpy() == label) & (lung_mask.numpy() > 0))
    print(f"Class {label}: {100*count/lung_voxels:.2f}% of mask; {count*voxel_volume:.1f} cubic physical units")
output_dir = Path("example_outputs/lung_functional_segmentation")
output_dir.mkdir(parents=True, exist_ok=True)
ants.image_write(labels_in_lung, str(output_dir / "labels_in_lung.nii.gz"))
ants.image_write(lung_mask, str(output_dir / "lung_mask.nii.gz"))
for label, probability in enumerate(probabilities):
    ants.image_write(probability, str(output_dir / f"probability_{label}.nii.gz"))
print("Outputs (matching files are overwritten):", output_dir.resolve())

Class fractions above report the model's numeric labels, including class 0
inside the supplied mask. They are not automatically validated ventilation
defect percentages. Interpret class definitions against the model's training
protocol and inspect mask coverage, intensity artifacts and class boundaries.